# Exploração dos Microdados do ENADE 2023

## Arquivos selecionados

O pacote do INEP contém 32 arquivos TXT. Para evitar leituras desnecessárias, esta etapa utilizará inicialmente apenas:

- `microdados2023_arq1.txt`: contém `CO_CURSO`, `CO_IES`, `CO_GRUPO`, `CO_MODALIDADE` e informações de localização do curso;
- `microdados2023_arq3.txt`: contém `CO_CURSO`, `TP_PRES` e `NT_GER`.

O arquivo `Dicionário_arquivos_variáveis_microdados_Enade_2023.xlsx` será utilizado para validar o significado das variáveis e dos códigos.

## Regra de relacionamento

Por causa da anonimização aplicada pelo INEP, os arquivos não podem ser relacionados pela posição das linhas nem no nível individual do estudante.
Cada arquivo será tratado separadamente, agregado no nível de curso e relacionado exclusivamente por `CO_CURSO`, conforme orientação do manual do ENADE 2023.

## Estratégia de processamento

Para reduzir o consumo de memória e melhorar o desempenho:

- somente os arquivos necessários serão carregados;
- o parâmetro `usecols` limitará a leitura às colunas utilizadas;
- as notas serão filtradas para `TP_PRES = 555`;
- os dados serão agregados por `CO_CURSO` antes do relacionamento;
- os arquivos originais permanecerão inalterados na camada Bronze.

# 01 - Imports e Caminhos dos Arquivos

**Objetivo:** preparar o ambiente do notebook — instalar bibliotecas e definir caminhos portáteis (sem depender de pastas de um computador específico) para localizar o dicionário de variáveis e os arquivos do ENADE 2023.

**Bibliotecas:**
- `pandas` — leitura e análise dos dados
- `openpyxl` — leitura do dicionário em Excel
- `pathlib` — caminhos portáteis e verificação dos arquivos

**Resultado esperado:** o notebook deve localizar corretamente três arquivos — o dicionário de variáveis, `microdados2023_arq1.txt` (cursos) e `microdados2023_arq3.txt` (presença e notas) — confirmando com três verificações que retornam `True`.

In [14]:
## 01. Caminhos

%pip install pandas openpyxl -q
%pip install duckdb -q
from pathlib import Path
import duckdb
import pandas as pd

pasta_atual = Path.cwd()

pasta_projeto = (
    pasta_atual.parent
    if pasta_atual.name == "notebooks"
    else pasta_atual
)

pasta_base = (
    pasta_projeto
    / "data"
    / "raw"
    / "microdados_enade_2023"
    / "Microdados_Enade_2023"
)

pasta_dados = pasta_base / "DADOS"

arquivo_dicionario = (
    pasta_base
    / "1.LEIA-ME"
    / "Dicionário_arquivos_variáveis_microdados_Enade_2023.xlsx"
)

arquivo_cursos = pasta_dados / "microdados2023_arq1.txt"
arquivo_notas = pasta_dados / "microdados2023_arq3.txt"

print(f"Dicionário encontrado: {arquivo_dicionario.exists()}")
print(f"Arquivo de cursos encontrado: {arquivo_cursos.exists()}")
print(f"Arquivo de notas encontrado: {arquivo_notas.exists()}")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Dicionário encontrado: True
Arquivo de cursos encontrado: True
Arquivo de notas encontrado: True


# 02 — Conferência do Dicionário de Dados

**Objetivo:** Ler o dicionário oficial dos Microdados do ENADE 2023 (aba `DICIONÁRIO_ARQUIVOS`) para identificar quais arquivos e variáveis são necessários para o desafio.

**Por quê:** Evita ler arquivos desnecessários, usar variáveis erradas, interpretar códigos incorretamente, gastar memória/tempo à toa e gera documentação da origem das variáveis escolhidas.

**Resultado esperado:** uma tabela com nome do arquivo, descrição e variáveis disponíveis, confirmando que:
- `microdados2023_arq1.txt` — dados de curso, IES, área e modalidade
- `microdados2023_arq3.txt` — dados de presença e nota geral
- variáveis necessárias: `CO_CURSO`, `CO_IES`, `CO_GRUPO`, `CO_MODALIDADE`, `TP_PRES`, `NT_GER`

In [2]:
dicionario_arquivos = pd.read_excel(
    arquivo_dicionario,
    sheet_name="DICIONÁRIO_ARQUIVOS"
)

dicionario_arquivos.head()

,Nome do arquivo,Informações,Variáveis
0,microdados2023_arq1,"Edição, código de curso e caracterização do cu...","NU_ANO, CO_CURSO, CO_IES, CO_CATEGAD, CO_ORGAC..."
1,microdados2023_arq2,"Edição, código de curso e informações acadêmic...","NU_ANO, CO_CURSO, ANO_FIM_EM, ANO_IN_GRAD, CO_..."
2,microdados2023_arq3,"Edição, código de curso e nº de itens válidos ...","NU_ANO, CO_CURSO, NU_ITEM_OFG, NU_ITEM_OFG_Z, ..."
3,microdados2023_arq4,"Edição, código de curso e avaliação dos estuda...","NU_ANO, CO_CURSO, QE_I27, QE_I28, QE_I29, QE_I..."
4,microdados2023_arq5,"Edição, código de curso e sexo","NU_ANO, CO_CURSO, TP_SEXO"


In [3]:
## 3.1. Selecionando as Colunas Necessárias de Cada Tabela: 
colunas_cursos = ["CO_CURSO","CO_IES","CO_GRUPO","CO_MODALIDADE","CO_MUNIC_CURSO","CO_UF_CURSO"]

colunas_notas = ["CO_CURSO","TP_PRES","NT_GER"]

## 3.2. Carregar Arquivos e Colunas Selecionadas
cursos = pd.read_csv(
    arquivo_cursos,
    sep=";",
    encoding="utf-8-sig",
    usecols=colunas_cursos
)

notas = pd.read_csv(
    arquivo_notas,
    sep=";",
    encoding="utf-8-sig",
    usecols=colunas_notas
)

## 3.3. Validar a Leitura:

print(f"Linhas em cursos: {len(cursos):,}")
print(f"Linhas em notas: {len(notas):,}")
print(f"Cursos únicos: {cursos['CO_CURSO'].nunique():,}")

display(cursos.head())
display(notas.head())


Linhas em cursos: 406,294
Linhas em notas: 406,294
Cursos únicos: 9,812


,CO_CURSO,CO_IES,CO_GRUPO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO
0,3,1,5710,1,5103403,51
1,3,1,5710,1,5103403,51
2,3,1,5710,1,5103403,51
3,3,1,5710,1,5103403,51
4,3,1,5710,1,5103403,51


,CO_CURSO,TP_PRES,NT_GER
0,1420197,222,NaN
1,1315386,222,NaN
2,1484333,222,NaN
3,1161015,222,NaN
4,17941,222,NaN


## 03 - Análise da Regra de relacionamento dos arquivos

Por causa da anonimização aplicada pelo INEP, os arquivos não podem ser relacionados pela posição das linhas ou no nível individual do estudante.
As informações serão agregadas separadamente e relacionadas apenas por `CO_CURSO`, conforme orientação do manual do ENADE.

# 04 - Definição da Modelagem Dimensional

#### Justificativa da Modelagem

O INEP define os códigos oficiais de relacionamento (`CO_CURSO`, `CO_IES`, `CO_GRUPO`, `CO_MODALIDADE`, `CO_MUNIC_CURSO`, `CO_UF_CURSO`), mas entrega os microdados do ENADE em formato flat, no nível de estudante, sem essa estrutura relacional pronta.

A `dim_curso` atua como hub: carrega as chaves estrangeiras que a ligam a `dim_ies`, `dim_grupo`, `dim_modalidade`, `dim_municipio` e `dim_uf` — tecnicamente um **floco de neve** no lado das dimensões — enquanto `fato_desempenho_curso` se relaciona apenas com `dim_curso`, via `CO_CURSO`, configurando uma **estrela** simples entre dimensão central e fato. O join sempre ocorre pelos códigos oficiais, agregados no nível de curso, nunca por posição de linha ou nível individual do estudante — preservando a anonimização dos microdados.

#### Dimensões a Serem Criadas

- **`dim_curso (dc)`:** código do curso e chaves para as demais dimensões  |   Relacionamento: `fdc[CO_CURSO] <> dc[CO_CURSO]`
- **`dim_ies (di)`:** código e nome da instituição.  |   Relacionamento: `dc[CO_IES] <> di[CO_IES]`
- **`dim_grupo (dg)`:** código e descrição da área avaliada.  |   Relacionamento: `dc[CO_GRUPO] <> dg[CO_GRUPO]`
- **`dim_modalidade (dmo)`:** código e descrição da modalidade — `0 = EaD` e `1 = Presencial`  |   Relacionamento: `dc[CO_MODALIDADE] <> dmo[CO_MODALIDADE]`
- **`dim_localizacao (dl)`:** código e nome do município, código e sigla da UF  |    Relacionamento: `dc[CO_MUNIC_CURSO] <> dl[CO_MUNIC_CURSO]
  
#### Tabela Fato a Ser Criada

- `fato_desempenho_curso (fdc)`: nota média e quantidade de estudantes avaliados por curso.  |   Relacionamento: `fdc[CO_CURSO] <> dc[CO_CURSO]`

#### Estrutura:

```
dim_ies ─────────────┐
dim_grupo ───────────┤
dim_modalidade ──────┼── dim_curso ─── fato_desempenho_curso
dim_localizacao ─────┘
```

# 05 — Preparação da Camada Silver

**Objetivo:** transformar os dados brutos da camada Bronze em tabelas organizadas no nível de curso. Os arquivos do ENADE estão no nível do estudante (o mesmo `CO_CURSO` aparece várias vezes) e foram ordenados de formas diferentes pelo INEP, por isso não podem ser relacionados pela posição das linhas.

**O que será feito:**
```
5.1) Reduzir os dados de curso para um registro por `CO_CURSO`
5.2) Filtrar as notas para estudantes presentes com resultado válido (`TP_PRES = 555`)
5.3) Agregar as notas por `CO_CURSO`
```

**Resultado esperado:** uma base com uma linha por curso, contendo código do curso, código da IES, área de avaliação, modalidade de ensino, localização, nota geral média e quantidade de estudantes avaliados.

## 5.1) Reduzir os dados de curso para um registro por CO_CURSO

Transformar os dados brutos da camada Bronze em uma base organizada no nível de curso, com uma linha por `CO_CURSO`.

#### Por que essa etapa é necessária:
Os arquivos estão no nível de estudante e foram ordenados de formas diferentes pelo INEP, então relacioná-los pela posição da linha misturaria dados de estudantes diferentes — um erro grave. Por isso, cada um é primeiro reduzido à granularidade de curso separadamente (deduplicação ou agregação) e só depois unidos por `merge` usando `CO_CURSO` como chave real.

#### Arquivos utilizados:

- **`microdados2023_arq1.txt`** (ENADE — curso, IES, área e modalidade): atributos descritivos do curso, repetidos para cada estudante do mesmo curso. Tratamento: **remoção de duplicatas** (`drop_duplicates` por `CO_CURSO`).
- **`microdados2023_arq3.txt`** (ENADE — presença e nota geral): métrica do curso — a nota (`NT_GER`) varia por estudante, por isso exige **agregação** (`groupby('CO_CURSO')` com média e contagem), filtrando antes só estudantes presentes com resultado válido (`TP_PRES = 555`).

#### Etapas:

I) **Deduplicar** os dados de curso, mantendo um único registro por `CO_CURSO`.

II) **Agregar** as notas dos estudantes por `CO_CURSO`, calculando a nota geral média e a quantidade de estudantes válidos.

III) **Relacionar** as duas tabelas resultantes usando `CO_CURSO` como chave, somente depois de ambas estarem na mesma granularidade.


#### Resultado esperado:

Uma base com uma linha por curso, contendo: código do curso, código da IES, área de avaliação, modalidade de ensino, localização, nota geral média e quantidade de estudantes avaliados.

In [4]:
##5.1.1. Validação Inicial:

print("Cursos")
print(f"Linhas: {len(cursos):,}")
print(f"Cursos únicos: {cursos['CO_CURSO'].nunique():,}")
print(f"CO_CURSO nulos: {cursos['CO_CURSO'].isna().sum():,}")

print("\nNotas")
print(f"Linhas: {len(notas):,}")
print(f"Cursos únicos: {notas['CO_CURSO'].nunique():,}")
print(f"NT_GER nulas: {notas['NT_GER'].isna().sum():,}")

Cursos
Linhas: 406,294
Cursos únicos: 9,812
CO_CURSO nulos: 0

Notas
Linhas: 406,294
Cursos únicos: 9,812
NT_GER nulas: 59,737


In [5]:
##5.1.2. Criar e Validar as Dimensões
colunas_dimensao = [
    "CO_CURSO",
    "CO_IES",
    "CO_GRUPO",
    "CO_MODALIDADE",
    "CO_MUNIC_CURSO",
    "CO_UF_CURSO"
]

#5.1.3. Verifica se um mesmo curso possui informações divergentes:
conflitos = (
    cursos
    .groupby("CO_CURSO")[colunas_dimensao[1:]]
    .nunique()
    .gt(1)
    .any(axis=1)
    .sum()
)

print(f"Cursos com informações conflitantes: {conflitos}")


Cursos com informações conflitantes: 0


In [6]:
#5.1.4. Criação da Dimensão Curso

## Criação do dim_cursos:

dim_cursos_raw = cursos[colunas_dimensao].drop_duplicates().sort_values("CO_CURSO").reset_index(drop=True)

## De-Para do Curso e Código Curso:
pasta_raw = pasta_projeto / "data" / "raw"

arquivos_encontrados = list(pasta_raw.rglob("MICRODADOS_CADASTRO_CURSOS_2023.CSV"))
arquivo_cadastro_cursos = arquivos_encontrados[0]

cadastro_cursos = pd.read_csv(arquivo_cadastro_cursos,sep=";", encoding="latin-1",usecols=["CO_CURSO","NO_CURSO"]).dropna(subset=["CO_CURSO"]).drop_duplicates(subset=["CO_CURSO"])

## Criação do dim_cursos já com o DE-PARA:
dim_cursos = (cursos[colunas_dimensao].drop_duplicates(subset=["CO_CURSO"]).merge(cadastro_cursos,on="CO_CURSO",how="left",validate="one_to_one")
    [
        [
            "CO_CURSO",
            "NO_CURSO",
            "CO_IES",
            "CO_GRUPO",
            "CO_MODALIDADE",
            "CO_MUNIC_CURSO",
            "CO_UF_CURSO"
        ]
    ]
    .sort_values("CO_CURSO").reset_index(drop=True)
)

dim_cursos.head(5)

,CO_CURSO,NO_CURSO,CO_IES,CO_GRUPO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO
0,3,Engenharia Civil,1,5710,1,5103403,51
1,9,Agronomia,1,17,1,5103403,51
2,10,Engenharia Florestal,1,6405,1,5103403,51
3,12,Medicina,1,12,1,5103403,51
4,16,Engenharia Elétrica,1,5806,1,5103403,51


In [7]:
#5.1.5. Criação da Dimensão IES

arquivos_ies_encontrados = list(pasta_raw.rglob("MICRODADOS_ED_SUP_IES_2023.CSV"))
arquivo_cadastro_ies = arquivos_ies_encontrados[0]

# Carrega somente as informações necessárias

cadastro_ies = pd.read_csv(arquivo_cadastro_ies, sep=";", encoding="latin-1", usecols=[ "CO_IES", "NO_IES","SG_IES"]).dropna(subset=["CO_IES"]).drop_duplicates(subset=["CO_IES"])

# Seleciona as IES presentes no ENADE e acrescenta nome e sigla

dim_ies = (dim_cursos_raw[["CO_IES"]].drop_duplicates().merge(cadastro_ies, on="CO_IES", how="left", validate="one_to_one")
    [[      "CO_IES",
            "NO_IES",
            "SG_IES"
     ]]
    .sort_values("CO_IES").reset_index(drop=True))

display(dim_ies.head())


,CO_IES,NO_IES,SG_IES
0,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,UFMT
1,2,UNIVERSIDADE DE BRASÍLIA,UNB
2,3,UNIVERSIDADE FEDERAL DE SERGIPE,UFS
3,4,UNIVERSIDADE FEDERAL DO AMAZONAS,UFAM
4,5,UNIVERSIDADE FEDERAL DO PIAUÍ,UFPI


In [8]:
#5.1.6. Criação da Dimensão Grupo/Área:

# Lê os códigos e descrições diretamente do dicionário do ENADE

mapa_grupos = pd.read_excel(arquivo_dicionario, sheet_name="DICIONÁRIO DE VARIÁVEIS", usecols="E", skiprows=27, nrows=28, header=None, names=["MAPEAMENTO"])

# Separar os campos CO_GRUPO do NO_GRUPO:

mapa_grupos[["CO_GRUPO", "NO_GRUPO"]] = (mapa_grupos["MAPEAMENTO"].str.split("=", n=1, expand=True))
mapa_grupos["CO_GRUPO"] = (pd.to_numeric(mapa_grupos["CO_GRUPO"].str.strip(),errors="coerce").astype("Int64"))
mapa_grupos["NO_GRUPO"] = (mapa_grupos["NO_GRUPO"].str.strip())
mapa_grupos = (mapa_grupos[["CO_GRUPO", "NO_GRUPO"]].dropna(subset=["CO_GRUPO"]).drop_duplicates(subset=["CO_GRUPO"]))   

#Criação da dim_grupo:

dim_grupo = (dim_cursos[["CO_GRUPO"]].drop_duplicates().merge(mapa_grupos,on="CO_GRUPO",how="left",validate="one_to_one")[["CO_GRUPO","NO_GRUPO"]].sort_values("CO_GRUPO").reset_index(drop=True))

dim_grupo.head(5)

,CO_GRUPO,NO_GRUPO
0,5,Medicina Veterinária
1,6,Odontologia
2,12,Medicina
3,17,Agronomia
4,19,Farmácia


In [9]:
#5.1.7. Criação da Dimensão Modalidade
dim_modalidade = (dim_cursos[["CO_MODALIDADE"]].drop_duplicates().sort_values("CO_MODALIDADE").reset_index(drop=True))
dim_modalidade["DS_MODALIDADE"] = (dim_modalidade["CO_MODALIDADE"].map({0: "EaD",1: "Presencial"}))

dim_modalidade.head(5)

,CO_MODALIDADE,DS_MODALIDADE
0,0,EaD
1,1,Presencial


In [10]:
#5.1.8. Criação da Dimensão Localização

# Lê os códigos e descrições diretamente do dicionário do ENADE
cadastro_municipios = pd.read_excel(arquivo_dicionario,sheet_name="MUNICÍPIOS",skiprows=3,usecols="B:D")
cadastro_municipios = (cadastro_municipios.rename(columns={"CÓDIGO DO MUNICÍPIO": "CO_MUNIC_CURSO","NOME DO MUNICÍPIO": "NO_MUNIC_CURSO","UF": "SG_UF"}))

# Padroniza o tipo do código para realizar o relacionamento

cadastro_municipios["CO_MUNIC_CURSO"] = pd.to_numeric(cadastro_municipios["CO_MUNIC_CURSO"],errors="coerce").astype("Int64")
cadastro_municipios =(cadastro_municipios.dropna(subset=["CO_MUNIC_CURSO"]).drop_duplicates(subset=["CO_MUNIC_CURSO"]))


# Mantém somente os municípios presentes nos cursos do ENADE

dim_localizacao = (dim_cursos[["CO_MUNIC_CURSO","CO_UF_CURSO"]]
    .drop_duplicates(subset=["CO_MUNIC_CURSO"])
    .merge(cadastro_municipios[["CO_MUNIC_CURSO", "NO_MUNIC_CURSO","SG_UF"]],on="CO_MUNIC_CURSO",how="left",validate="one_to_one")
    [["CO_MUNIC_CURSO","NO_MUNIC_CURSO","CO_UF_CURSO","SG_UF"]]
    .sort_values("CO_MUNIC_CURSO").reset_index(drop=True))

dim_localizacao.head(5)



,CO_MUNIC_CURSO,NO_MUNIC_CURSO,CO_UF_CURSO,SG_UF
0,1100023,ARIQUEMES,11,RO
1,1100049,CACOAL,11,RO
2,1100064,COLORADO DO OESTE,11,RO
3,1100114,JARU,11,RO
4,1100122,JI-PARANA,11,RO


In [11]:
print("\nDimensões criadas:")
print(f"dim_cursos: {len(dim_cursos):,} registros")
print(f"dim_ies: {len(dim_ies):,} registros")
print(f"dim_grupo: {len(dim_grupo):,} registros")
print(f"dim_modalidade: {len(dim_modalidade):,} registros")
print(f"dim_localizacao: {len(dim_localizacao):,} registros")


Dimensões criadas:
dim_cursos: 9,812 registros
dim_ies: 1,347 registros
dim_grupo: 28 registros
dim_modalidade: 2 registros
dim_localizacao: 718 registros


## 5.2) Filtrar as notas para estudantes presentes com resultado válido (`TP_PRES = 555`)

In [12]:
#5.2.1.Filtra estudantes presentes com resultado válido
notas_validas = notas.loc[(notas["TP_PRES"] == 555) & notas["NT_GER"].notna()].copy()


## 5.3) Agregar as notas por `CO_CURSO`

In [13]:
#5.3.1. Cria a tabela fato no nível de curso
fato_desempenho_curso = (notas_validas.groupby("CO_CURSO", as_index=False).agg(NT_GER_MEDIA=("NT_GER", "mean"),QT_AVALIADOS=("NT_GER", "size")))
fato_desempenho_curso["NT_GER_MEDIA"] = (fato_desempenho_curso["NT_GER_MEDIA"].round(2))

#5.3.2. Valida a unicidade das chaves
if not dim_cursos["CO_CURSO"].is_unique:
    raise ValueError(
        "A dimensão de cursos possui CO_CURSO duplicado.")

if not fato_desempenho_curso["CO_CURSO"].is_unique:
    raise ValueError(
        "A tabela fato possui CO_CURSO duplicado.")

#5.3.3. Valida a integridade referencial:
cursos_sem_dimensao = fato_desempenho_curso.loc[~fato_desempenho_curso["CO_CURSO"].isin(dim_cursos["CO_CURSO"])]

if not cursos_sem_dimensao.empty:
    raise ValueError(
        f"{len(cursos_sem_dimensao)} cursos da tabela fato "
        "não foram encontrados na dimensão de cursos.")

#5.3.5. Identifica cursos sem nota válida
cursos_sem_nota = dim_cursos.loc[~dim_cursos["CO_CURSO"].isin(fato_desempenho_curso["CO_CURSO"])]

print(f"Cursos na dimensão: {len(dim_cursos):,}")
print(
    "Cursos na tabela fato:",
    f"{len(fato_desempenho_curso):,}"
)
print(
    "Cursos da dimensão sem nota válida:",
    f"{len(cursos_sem_nota):,}")
print(
    "Cursos da fato sem dimensão:",
    f"{len(cursos_sem_dimensao):,}")

fato_desempenho_curso.head(5)

Cursos na dimensão: 9,812
Cursos na tabela fato: 9,380
Cursos da dimensão sem nota válida: 432
Cursos da fato sem dimensão: 0


,CO_CURSO,NT_GER_MEDIA,QT_AVALIADOS
0,3,59.61,31
1,9,59.38,36
2,10,45.71,11
3,12,69.44,78
4,16,51.27,23


# 6 — Criação do Banco em Modelo Medalhão:


In [16]:
## Criar Pasta e o Banco:
pasta_banco = pasta_projeto / "data" / "database"
pasta_banco.mkdir(parents=True,exist_ok=True)

arquivo_banco = pasta_banco / "enade.duckdb"

conexao = duckdb.connect(str(arquivo_banco))


Banco criado em: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\database\enade.duckdb


In [18]:
##Criar Camadas:
conexao.execute("CREATE SCHEMA IF NOT EXISTS bronze")
conexao.execute("CREATE SCHEMA IF NOT EXISTS silver")
conexao.execute("CREATE SCHEMA IF NOT EXISTS gold")

#Conferência da Criação:
schemas_criados = conexao.execute(
"""
SELECT schema_name
FROM information_schema.schemata
WHERE schema_name IN ('bronze', 'silver', 'gold')
ORDER BY schema_name
"""
).df()

display(schemas_criados)

,schema_name
0,bronze
1,gold
2,silver


## 6.1 — Carga da Camada Bronze

A camada Bronze armazena os arquivos originais do ENADE sem transformação. Todas as colunas e linhas são preservadas para garantir rastreabilidade até a fonte oficial.
Os dados serão armazenados inicialmente como texto, evitando alterações automáticas de tipo durante a carga.

### 6.1.1. Dados Enade:

In [23]:
## Caminhos:
caminho_cursos_bronze = ( arquivo_cursos.as_posix().replace("'", "''") )
caminho_notas_bronze = ( arquivo_notas.as_posix().replace("'", "''") )

##Criação ou Substituição das Tabelas do ENADE:
conexao.execute( f""" CREATE OR REPLACE TABLE bronze.enade_cursos_raw AS SELECT * FROM read_csv_auto( '{caminho_cursos_bronze}', delim = ';', header = true, all_varchar = true ) """ )
conexao.execute( f""" CREATE OR REPLACE TABLE bronze.enade_notas_raw AS SELECT * FROM read_csv_auto( '{caminho_notas_bronze}', delim = ';', header = true, all_varchar = true ) """ )

##Validação das Tabelas ENADE:
validacao_bronze = conexao.execute( 
    """ SELECT 'enade_cursos_raw' AS TABELA, COUNT(*) AS QT_REGISTROS FROM bronze.enade_cursos_raw
        UNION ALL
        SELECT 'enade_notas_raw' AS TABELA, COUNT(*) AS QT_REGISTROS FROM bronze.enade_notas_raw """ ).df()

validacao_bronze.head(5)

,TABELA,QT_REGISTROS
0,enade_cursos_raw,406294
1,enade_notas_raw,406294


In [24]:
## Caminhos:
caminho_censo_cursos_bronze = ( arquivo_cadastro_cursos .as_posix() .replace("'", "''") )
caminho_censo_ies_bronze = ( arquivo_cadastro_ies .as_posix() .replace("'", "''") )

##Criação ou Substituição das Tabelas do CENSO:
conexao.execute( f""" CREATE OR REPLACE TABLE bronze.censo_cursos_raw AS SELECT * FROM read_csv_auto( '{caminho_censo_cursos_bronze}', delim = ';', header = true, encoding = 'latin-1', all_varchar = true ) """ )
conexao.execute( f""" CREATE OR REPLACE TABLE bronze.censo_ies_raw AS SELECT * FROM read_csv_auto( '{caminho_censo_ies_bronze}', delim = ';', header = true, encoding = 'latin-1', all_varchar = true ) """ )

##Validação das Tabelas CENSO:
validacao_bronze = conexao.execute( 
    """ SELECT 'enade_cursos_raw' AS TABELA, COUNT(*) AS QT_REGISTROS FROM bronze.enade_cursos_raw
    UNION ALL
    SELECT 'enade_notas_raw', COUNT(*) FROM bronze.enade_notas_raw
    UNION ALL
    SELECT 'censo_cursos_raw', COUNT(*) FROM bronze.censo_cursos_raw
    UNION ALL
    SELECT 'censo_ies_raw', COUNT(*) FROM bronze.censo_ies_raw
    ORDER BY TABELA """ ).df()

validacao_bronze.head(5)

,TABELA,QT_REGISTROS
0,censo_cursos_raw,671610
1,censo_ies_raw,2580
2,enade_cursos_raw,406294
3,enade_notas_raw,406294


## 6.2 — Carga da Camada Bronze

A camada Silver contém os dados selecionados e tratados para utilização no modelo dimensional.

Nesta etapa:
- Os cursos são reduzidos para um registro por `CO_CURSO`;
- São mantidos somente estudantes presentes (`TP_PRES = 555`);
- Registros com `NT_GER` nula são removidos antes da agregação;
- As notas ainda permanecem no nível original, sem cálculo de média.

### 6.2.1. Dados Enade:

In [ ]:
 Gold

**Objetivo:** reunir a dimensão de cursos, as dimensões descritivas e a tabela fato em uma única base analítica.

**Por que é necessário:** o modelo dimensional preserva a organização e a integridade dos dados, enquanto a base analítica simplifica as consultas que responderão as perguntas que podem surgir o desafio.

**Resultado esperado:** uma linha por curso, contendo seus atributos descritivos, localização, nota média e quantidade de estudantes avaliados.

In [ ]:
##6.1. dim_municipios:

dim_uf = (
    dim_municipio[
        [
            "CO_UF_CURSO",
            "SG_UF"
        ]
    ]
    .drop_duplicates()
    .sort_values("CO_UF_CURSO")
    .reset_index(drop=True)
)

display(dim_municipio.head())
display(dim_uf.head())